# Scaling the MNIST MLP

> We scale a neural network by freezing it and adding new trainable layers.

#### Imports and Setup

In [ ]:
from ast import literal_eval
from copy import deepcopy

import graphviz
import matrepr
import torch
import torchinfo
from safetensors.torch import load_model, save_model
from torch import nn
from torch.nn import (
    HuberLoss,
)
from torch.nn import functional as F
from torch.optim import SGD, Adam, AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from learningdynamics.models import MLP, init_weights
from learningdynamics.utils import (
    compute_model_accuracy,
    getActivation,
    safetensors_metadata_parser,
)

In [ ]:
# | hide
matrepr.params.title = True
matrepr.params.indices = False
matrepr.params.precision = 4
matrepr.cell_align = "left"
graphviz.set_jupyter_format("png")

%matplotlib inline

#### Load model trained on binary classification.

In [ ]:
file_path = "../saved_weights/mnist_model.safetensors"

# Parse metadata
metadata = safetensors_metadata_parser(file_path=file_path)

# Load model
model = MLP(
    config=literal_eval(metadata["config"]),
    nonlinearity=metadata["nonlinearity"],
    in_features=784,
    out_features=10,
)
_ = load_model(model, file_path, device="cuda:0")

#### Setup the dataset

In [ ]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),  # first, convert image to PyTorch tensor
        transforms.Normalize((0.1307,), (0.3081,)),  # normalize inputs
    ]
)
train_set = datasets.MNIST(
    root="/mnt/datasets/", train=True, download=True, transform=transform
)
test_set = datasets.MNIST(
    root="/mnt/datasets/", train=False, download=True, transform=transform
)

#### Show model and test accuracy

In [ ]:
print(f"Testing Accuracy: {compute_model_accuracy(model, test_set)}")

Testing Accuracy: 0.972000002861023


In [ ]:
torchinfo.summary(model, row_settings=["var_names", "ascii_only"])

Layer (type (var_name))                  Param #
MLP (MLP)                                --
+ Sequential (features)                  --
|    + Linear (0)                        200,960
|    + ReLU (1)                          --
|    + Linear (2)                        32,896
|    + ReLU (3)                          --
|    + Linear (4)                        8,256
|    + ReLU (5)                          --
|    + Linear (6)                        2,080
|    + ReLU (7)                          --
|    + Linear (8)                        330
Total params: 244,522
Trainable params: 244,522
Non-trainable params: 0

#### Scale Model

In [ ]:
def freeze_and_scale_model(
    model,
    start,
    layers_to_add=1,
    device="cuda:0",
):
    # Set up scaled model
    model.to(device)
    scaled_model = deepcopy(model)

    # Update model
    scaled_container = torch.nn.Sequential()
    for idx, module in enumerate(scaled_model.features):
        # Copy over weights and biases
        if hasattr(module, "weight"):
            module.weight.requires_grad_(False)
        if hasattr(module, "bias"):
            module.bias.requires_grad_(False)
        scaled_container.append(module)
        # Add new layers
        if idx == start:
            prev_out = model.features[idx - 1].out_features
            for _ in range(layers_to_add):
                linear_layer = nn.Linear(prev_out, prev_out)
                linear_layer.apply(init_weights)
                scaled_container.append(linear_layer)
                scaled_container.append(nn.ReLU())
    scaled_model.features = scaled_container
    scaled_model.to(device=device)

    scaled_model.to(device="cpu")
    return scaled_model

#### Control panel

In [ ]:
LAYER_START = 5

In [ ]:
DEVICE = "cuda:0"
PRINT_FREQ = 1

NUM_LAYERS_ADDED = 10
DIMENSIONALITY = model.features[LAYER_START + 1].in_features
LAYER_END = LAYER_START + 2

BATCH_SIZE = 4_096
NUM_EPOCHS = 100
LEARNING_RATE = 3e-3
BETAS = (0.9, 0.99)

dataloader = DataLoader(dataset=train_set, shuffle=True, batch_size=BATCH_SIZE)

In [ ]:
scaled_model = freeze_and_scale_model(
    model,
    start=LAYER_START,
    layers_to_add=NUM_LAYERS_ADDED,
)

In [ ]:
torchinfo.summary(scaled_model, row_settings=["var_names", "ascii_only"])
# scaled_model

Layer (type (var_name))                  Param #
MLP (MLP)                                --
+ Sequential (features)                  --
|    + Linear (0)                        (200,960)
|    + ReLU (1)                          --
|    + Linear (2)                        (32,896)
|    + ReLU (3)                          --
|    + Linear (4)                        (8,256)
|    + ReLU (5)                          --
|    + Linear (6)                        4,160
|    + ReLU (7)                          --
|    + Linear (8)                        4,160
|    + ReLU (9)                          --
|    + Linear (10)                       4,160
|    + ReLU (11)                         --
|    + Linear (12)                       4,160
|    + ReLU (13)                         --
|    + Linear (14)                       4,160
|    + ReLU (15)                         --
|    + Linear (16)                       4,160
|    + ReLU (17)                         --
|    + Linear (18)                 

In [ ]:
print(f"Testing Accuracy: {compute_model_accuracy(scaled_model.to('cpu'), test_set)}")

Testing Accuracy: 0.06599999964237213


#### Train scaled model

In [ ]:
activation_dict = {}
hooks = []

In [ ]:
# Setup training
huber_criterion = HuberLoss(delta=1)
optimizer = AdamW(scaled_model.parameters(), lr=LEARNING_RATE, betas=BETAS)
scheduler = ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, threshold=0.001, verbose=True
)

In [ ]:
scaled_model.to(DEVICE)
scaled_model.train()

# Attach hook to original
hooks.append(
    model.features[LAYER_END].register_forward_hook(
        getActivation(f"layer_{LAYER_END}", activation_dict)
    )
)

# Attach hook to scaled
new_end = LAYER_END + NUM_LAYERS_ADDED * 2
hooks.append(
    scaled_model.features[new_end].register_forward_hook(
        getActivation(f"scaled_layer_{new_end}", activation_dict, detach=False)
    )
)

# Run training
loss_vector = []
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0  # Track loss within the epoch

    # Iterate through dataloader
    for input, label in dataloader:
        # Feed input in to get activations
        input, label = input.to(DEVICE), torch.flatten(label.to(DEVICE))
        output = scaled_model(input)

        with torch.no_grad():
            _ = model(input)

        # Zero out optimizer
        optimizer.zero_grad()

        # Grab activations
        output_activation = activation_dict[f"scaled_layer_{new_end}"].to(DEVICE)
        target_activation = activation_dict[f"layer_{LAYER_END}"].to(DEVICE)

        huber_loss = huber_criterion(output_activation, target_activation)

        loss = huber_loss
        loss.backward()
        optimizer.step()

        running_loss += loss.item()  # Accumulate loss
        
    epoch_loss = running_loss / len(dataloader)  # Average loss over all batches
    loss_vector.append(epoch_loss)  # Store loss for later analysis

    # Step the scheduler
    scheduler.step(epoch_loss)

    epoch_loss = running_loss / len(dataloader)  # Average loss over all batches
    loss_vector.append(epoch_loss)  # Store loss for later analysis5

    if (epoch + 1) % PRINT_FREQ == 0:
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS}, Loss: {epoch_loss:.4f}")

# Remove hook
for hook in hooks:
    hook.remove()

Epoch 1/100, Loss: 2.0224
Epoch 2/100, Loss: 1.2512
Epoch 3/100, Loss: 0.9209
Epoch 4/100, Loss: 0.6978
Epoch 5/100, Loss: 0.5813
Epoch 6/100, Loss: 0.4987
Epoch 7/100, Loss: 0.4847
Epoch 8/100, Loss: 0.4305
Epoch 9/100, Loss: 0.4050
Epoch 10/100, Loss: 0.4035
Epoch 11/100, Loss: 0.3687
Epoch 12/100, Loss: 0.3653
Epoch 13/100, Loss: 0.3456
Epoch 14/100, Loss: 0.3365
Epoch 15/100, Loss: 0.3255
Epoch 16/100, Loss: 0.3154
Epoch 17/100, Loss: 0.3115
Epoch 18/100, Loss: 0.2952
Epoch 19/100, Loss: 0.2927
Epoch 20/100, Loss: 0.2850
Epoch 21/100, Loss: 0.2725
Epoch 22/100, Loss: 0.2715
Epoch 23/100, Loss: 0.2628
Epoch 24/100, Loss: 0.2556
Epoch 25/100, Loss: 0.2505
Epoch 26/100, Loss: 0.2514
Epoch 27/100, Loss: 0.2380
Epoch 28/100, Loss: 0.2439
Epoch 29/100, Loss: 0.2309
Epoch 30/100, Loss: 0.2278
Epoch 31/100, Loss: 0.2288
Epoch 32/100, Loss: 0.2219
Epoch 33/100, Loss: 0.2222
Epoch 34/100, Loss: 0.2103
Epoch 35/100, Loss: 0.2153
Epoch 36/100, Loss: 0.2083
Epoch 37/100, Loss: 0.1995
Epoch 38/1

In [ ]:
print(
    f"Testing Accuracy: {compute_model_accuracy(scaled_model.to('cpu').eval(), test_set)}"
)

Testing Accuracy: 0.9682999849319458


#### Update config

In [ ]:
config = literal_eval(metadata["config"])
insert = [model.features[LAYER_START - 1].out_features] * NUM_LAYERS_ADDED
config[LAYER_START // 2 : LAYER_START // 2] = insert

In [ ]:
metadata_dict = {
    "config": str(config),
    "dataset": metadata["dataset"],
    "nonlinearity": metadata["nonlinearity"],
    "added_layers": str(NUM_LAYERS_ADDED),
    "layer_start": str(LAYER_START),
    "layer_in_out": str(DIMENSIONALITY),
}

save_model(
    scaled_model,
    f"../saved_weights/mnist_start{LAYER_START}_scaled_model.safetensors",
    metadata=metadata_dict,
)